# Minimal Coding Agent

Let's build the smallest coding agent that is still useful.

The whole idea is just a loop:

1. Send the conversation to the model.
2. If the model gives us a bash command, run it.
3. Send the command output back to the model.
4. Stop when the model replies without a bash command.

That is the agent.

This notebook is intentionally small. It is a teaching example, not a production system.

## What we are using

Only the Python standard library:

- `urllib.request` to call the OpenAI API
- `json` to build and parse payloads
- `subprocess` to run bash
- `re` to find a bash block in the model output
- `os` for `OPENAI_API_KEY`

We are not using the OpenAI Python SDK here on purpose. The point is to make every moving part visible.

In [ ]:
import json
import os
import re
import subprocess
import urllib.error
import urllib.request


## Step 1: one tiny function to call OpenAI

We will use the Responses API. OpenAI recommends Responses for new projects, and it works fine over plain HTTP.

We only depend on `OPENAI_API_KEY`. No SDK config, no extra secrets.

In [ ]:
API_URL = "https://api.openai.com/v1/responses"
MODEL = "gpt-5-mini"


def call_openai(messages, model=MODEL):
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("Set OPENAI_API_KEY before calling the agent.")

    payload = {
        "model": model,
        "input": messages,
        "store": False,
    }

    request = urllib.request.Request(
        API_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        body = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"OpenAI API error {error.code}: {body}") from error


## Step 2: pull the assistant text out of the JSON

The raw response is nested. We only want the plain text the assistant produced.

In [ ]:
def response_text(response_json):
    parts = []

    for item in response_json.get("output", []):
        if item.get("type") != "message":
            continue

        for content in item.get("content", []):
            if content.get("type") == "output_text":
                parts.append(content.get("text", ""))

    return "".join(parts).strip()


## Step 3: define a tiny protocol

We are not doing formal tool calling in this notebook.

Instead, we tell the model:

- If you need to inspect or change the repo, reply with exactly one fenced `bash` block.
- If you are done, reply in plain text with no fenced code block.

That gives us a stop condition that is trivial to parse.

In [ ]:
SYSTEM_PROMPT = """You are a tiny coding agent.

When you need to run a shell command, reply with exactly one fenced bash block and nothing else.

Example:
```bash
ls
```

When you are done, reply with plain text and do not include a bash block.
Keep commands small and relevant to the task.
"""

BASH_BLOCK_RE = re.compile(r"```bash\n(.*?)```", re.DOTALL)


def extract_bash_command(text):
    match = BASH_BLOCK_RE.search(text)
    if not match:
        return None
    return match.group(1).strip()


## Step 4: run bash and package the result

We will send stdout, stderr, and the exit code back to the model.

One practical detail: clip long output so the next prompt does not blow up.

In [ ]:
def clip(text, limit=4000):
    if len(text) <= limit:
        return text
    return text[:limit] + "\n...<truncated>"


def run_bash(command, cwd=None):
    completed = subprocess.run(
        ["bash", "-lc", command],
        cwd=cwd,
        capture_output=True,
        text=True,
        timeout=30,
    )
    return {
        "returncode": completed.returncode,
        "stdout": clip(completed.stdout),
        "stderr": clip(completed.stderr),
    }


def format_bash_result(command, result):
    return f"""Command:
{command}

Exit code: {result['returncode']}

Stdout:
{result['stdout']}

Stderr:
{result['stderr']}
"""


## Step 5: the agent loop

This is the whole thing.

We keep a normal conversation history. The only special rule is that if the assistant gives us a bash block, we execute it and feed the result back in as the next user message.

In [ ]:
def run_agent(task, cwd=None, max_steps=8):
    cwd = cwd or os.getcwd()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Working directory: {cwd}\n\nTask: {task}",
        },
    ]

    for step in range(1, max_steps + 1):
        response_json = call_openai(messages)
        assistant_message = response_text(response_json)

        print(f"\nStep {step} assistant:\n")
        print(assistant_message)

        messages.append({"role": "assistant", "content": assistant_message})

        command = extract_bash_command(assistant_message)
        if command is None:
            print("\nNo bash block found. Stopping.")
            return assistant_message

        print(f"\n$ {command}\n")
        result = run_bash(command, cwd=cwd)

        if result["stdout"]:
            print(result["stdout"])
        if result["stderr"]:
            print(result["stderr"])

        messages.append(
            {"role": "user", "content": format_bash_result(command, result)}
        )

    raise RuntimeError("Agent hit max_steps before finishing.")


## Try it

Start with a tiny task. You want something that makes the loop obvious.

I would not start with file editing yet. Start with inspection.

In [ ]:
task = "List the files in this project, then tell me in one short paragraph what kind of project this is."

# Uncomment the next line when OPENAI_API_KEY is set.
# run_agent(task)


## What matters here

- The agent is mostly a loop, not a framework.
- The critical design choice is the output protocol.
- Bash output becomes the model's next context.
- The stop condition is simple: no bash block, so we stop.

## What this notebook does not do

- No sandboxing
- No retries
- No diff-aware editing
- No streaming
- No formal tool schemas

That is fine for a first notebook. If you can explain this version clearly, you can build the safer version later.

## Official docs behind the wire format

- API auth overview: https://developers.openai.com/api/reference/overview
- Responses is recommended for new projects: https://developers.openai.com/api/docs/guides/migrate-to-responses
- Current model list: https://platform.openai.com/docs/models
